# Data processing

This notebook turns a directory of raw daily SPX option-chain CSVs into one
Parquet file per month, keeping only the chains the analysis uses, and
downloads the Cboe SET settlement values that the forecasts are scored
against. The analysis itself lives in `02_project.ipynb`.

The raw data are not redistributed in this repository. Everything below
assumes `data/raw/SPX/<year>/*.csv` exists locally.

## What the cleaning does, and why

**One quote per option key.** The vendor files contain groups of rows that
share an `OptionKey` but carry different quotes. These cannot be told apart
after the fact, so every row in such a group is discarded rather than one
being picked arbitrarily. The most likely cause is the introduction of
PM-settled monthly (SPXW) series that the vendor did not separate from the
AM-settled standard monthlies; since the two settle against different prints,
mixing them would put the wrong realised value against a chain.

**Standard monthlies only.** Only third-Friday expirations are retained.
Those settle against SET, the Cboe special opening quotation, which is the
realised value used for scoring in the analysis notebook. Weeklies and
end-of-month series have no published SET, so including them would mean
scoring against a different quantity.

**Three horizons.** For each observation date the nearest distinct expiration
to each of 30, 60 and 90 calendar days is kept, provided it is within ten
days of the target. Exact horizons rarely exist for monthly expirations. The
realised distribution of days to expiry within each bucket is reported at the
end of this notebook, because it is wide enough to matter: CRPS scales
roughly with the square root of time, so a bucket spanning twenty to forty
days spans about a forty per cent range in expected score.

**Time to expiry.** Standard monthlies are AM-settled: the settlement print
comes from Friday's opening auction, so the option effectively stops trading
at Thursday's close. One calendar day is subtracted for this, and the year
fraction uses a 365-day convention.

**Quote validity.** A quote is valid when the ask is positive and at least the
bid; it is two-sided when the bid is also positive. Mid-prices are only
computed for valid quotes -- an invalid one gets `NaN` rather than a
misleading midpoint. Rows failing validity are written to a companion
`_invalid.parquet` file rather than dropped silently, so the counts can be
audited.

## Which years survive

`config.ANALYSIS_YEARS` is not contiguous, and an unexplained gap -- 2020 in
particular, which removes the most informative regime in the sample for tail
behaviour -- is the first thing a reader will ask about.

The exclusion rule is a number rather than a judgement call:

> A year is dropped when the mean monthly share of rows sharing an
> `option_key` with a differing quote exceeds `config.MAX_DUPLICATE_KEY_SHARE`.

Once a large fraction of a year's rows cannot be attributed to a single
settlement convention, the chains built from what remains are not
trustworthy: the most likely cause is PM-settled SPXW series that the vendor
did not separate from the AM-settled standard monthlies, and since the two
settle against different prints, mixing them puts the wrong realised value
against a chain.

The table below is **generated from the coverage numbers**, not written by
hand, so it cannot drift away from the data or be left half-filled.

In [ ]:
import json
import re
import time
from collections import defaultdict
from io import StringIO
from pathlib import Path
from urllib.request import urlopen

import numpy as np
import pandas as pd
from pandas.tseries.holiday import GoodFriday

from implied_stock_distributions.config import (
    AM_SETTLEMENT_DAY_ADJUSTMENT,
    ANALYSIS_YEARS,
    CHAIN_CATALOG_PATH,
    DAY_COUNT_DAYS_PER_YEAR,
    EXCLUDED_YEARS,
    MINIMUM_DTE_DAYS,
    PROCESSED_DIR,
    SET_DOWNLOAD_YEARS,
    SET_VALUES_PATH,
    SPX_PROCESSED_DIR,
    SPX_RAW_DIR,
    STANDARD_MONTHLY_ONLY,
    TARGET_DTES,
    TARGET_DTE_TOLERANCE_DAYS,
)

OVERWRITE = True
VALID_SYMBOLS = {"SPX"}

## Reading and validating one daily file

In [ ]:
REQUIRED_COLUMNS = {
    "OptionKey", "Symbol", "ExpirationDate", "AskPrice", "AskSize",
    "BidPrice", "BidSize", "LastPrice", "PutCall", "StrikePrice",
    "Volume", "OpenInterest", "UnderlyingPrice", "DataDate",
}

COLUMN_NAMES = {
    "OptionKey": "option_key", "Symbol": "symbol",
    "ExpirationDate": "expiration_date", "AskPrice": "ask_price",
    "AskSize": "ask_size", "BidPrice": "bid_price", "BidSize": "bid_size",
    "LastPrice": "last_price", "PutCall": "put_call",
    "StrikePrice": "strike_price", "Volume": "volume",
    "OpenInterest": "open_interest", "UnderlyingPrice": "underlying_price",
    "DataDate": "data_date",
}

READ_DTYPES = {
    "OptionKey": "string", "Symbol": "string", "AskPrice": "float64",
    "AskSize": "Int64", "BidPrice": "float64", "BidSize": "Int64",
    "LastPrice": "float64", "PutCall": "string", "StrikePrice": "float64",
    "Volume": "Int64", "OpenInterest": "Int64", "UnderlyingPrice": "float64",
}

DAILY_COLUMNS = [
    "option_key", "symbol", "data_date", "expiration_date",
    "settlement_date", "time_to_expiry_years", "put_call", "strike_price",
    "underlying_price", "bid_price", "ask_price", "mid_price", "bid_size",
    "ask_size", "last_price", "volume", "open_interest", "dte_calendar_days",
    "is_standard_monthly", "quote_is_valid", "quote_is_two_sided",
    "source_file",
]

OUTPUT_COLUMNS = DAILY_COLUMNS + ["target_dte_days"]


class SkipDailyFile(ValueError):
    """Raised when a daily file cannot contribute any usable rows."""


def parse_date_column(series: pd.Series) -> pd.Series:
    """Parse both date formats present in the source files."""
    parsed = pd.to_datetime(series, format="%Y-%m-%d", errors="coerce")
    unresolved = parsed.isna()
    parsed.loc[unresolved] = pd.to_datetime(
        series.loc[unresolved], format="%m/%d/%Y", errors="coerce"
    )
    return parsed


def is_standard_monthly_expiration(series: pd.Series) -> pd.Series:
    """Identify third-Friday expirations under both date conventions.

    Older files record the Saturday following the third Friday; newer ones
    record the Friday itself. A Thursday is allowed for the occasional
    holiday-adjusted expiration. The option root would be a better
    identifier where it is available.
    """
    dates = pd.to_datetime(series)
    weekday, day = dates.dt.dayofweek, dates.dt.day

    return (
        ((weekday == 4) & day.between(15, 21))
        | ((weekday == 5) & day.between(16, 22))
        | ((weekday == 3) & day.between(14, 20))
    )


def standard_spx_settlement_date(expiration_dates: pd.Series) -> pd.Series:
    """SET date for standard monthly AM-settled SPX options."""
    dates = pd.to_datetime(expiration_dates)
    month_start = dates.dt.to_period("M").dt.to_timestamp()

    days_to_first_friday = (4 - month_start.dt.dayofweek) % 7
    third_friday = month_start + pd.to_timedelta(days_to_first_friday + 14, unit="D")

    if third_friday.notna().any():
        good_fridays = GoodFriday.dates(third_friday.min(), third_friday.max())
        third_friday = third_friday.mask(
            third_friday.isin(good_fridays), third_friday - pd.Timedelta(days=1)
        )

    return third_friday

In [ ]:
def inspect_file_data_date(path: Path) -> pd.Timestamp:
    """Read only DataDate and return the most common valid date in a CSV."""
    try:
        text = pd.read_csv(
            path, usecols=["DataDate"], dtype={"DataDate": "string"}
        )["DataDate"]
    except ValueError as exc:
        raise ValueError(f"{path.name}: missing DataDate column") from exc

    dates = parse_date_column(text).dropna().dt.normalize()
    if dates.empty:
        raise ValueError(f"{path.name}: no valid DataDate values")

    counts = dates.value_counts()
    if len(counts) > 1 and counts.iloc[0] == counts.iloc[1]:
        raise ValueError(f"{path.name}: cannot determine a single data date")

    return pd.Timestamp(counts.index[0])


def discover_files_by_month(raw_dir: Path) -> dict:
    """Group daily CSVs by the calendar month of their data date."""
    if not raw_dir.exists():
        raise FileNotFoundError(f"Raw data directory not found: {raw_dir}")

    files_by_month = defaultdict(list)

    for year_dir in sorted(raw_dir.iterdir()):
        if not year_dir.is_dir():
            continue
        try:
            folder_year = int(year_dir.name)
        except ValueError:
            print(f"Warning: skipping non-year directory {year_dir}")
            continue

        for path in sorted(year_dir.glob("*.csv")):
            data_date = inspect_file_data_date(path)
            if data_date.year != folder_year:
                print(
                    f"Warning: {path.name}: DataDate year {data_date.year} "
                    f"does not match directory {folder_year}"
                )
            files_by_month[(int(data_date.year), int(data_date.month))].append(
                (path, data_date)
            )

    return dict(files_by_month)

In [ ]:
def read_and_prepare_daily_file(
    path: Path, expected_data_date: pd.Timestamp
) -> pd.DataFrame:
    """Read, validate, standardise and enrich one daily CSV."""
    raw = pd.read_csv(path, dtype=READ_DTYPES)

    missing = REQUIRED_COLUMNS.difference(raw.columns)
    if missing:
        raise ValueError(f"{path.name}: missing columns {sorted(missing)}")

    # Normalise before filtering, so values such as " spx" are retained.
    raw["Symbol"] = raw["Symbol"].str.strip().str.upper()
    valid_symbol = raw["Symbol"].isin(VALID_SYMBOLS)

    if not valid_symbol.all():
        others = sorted(raw.loc[~valid_symbol, "Symbol"].dropna().unique().tolist())
        print(
            f"Warning: {path.name}: discarding {int((~valid_symbol).sum())} "
            f"rows with symbols {others}"
        )
        raw = raw.loc[valid_symbol].copy()

    if raw.empty:
        raise SkipDailyFile(f"{path.name}: no SPX rows remain")

    df = raw.loc[:, list(COLUMN_NAMES)].rename(columns=COLUMN_NAMES).copy()
    df["data_date"] = parse_date_column(df["data_date"])
    df["expiration_date"] = parse_date_column(df["expiration_date"])
    df["symbol"] = df["symbol"].str.strip().str.upper()
    df["put_call"] = df["put_call"].str.strip().str.lower()

    bad_dates = df[["data_date", "expiration_date"]].isna().any(axis=1)
    if bad_dates.any():
        print(f"Warning: {path.name}: discarding {int(bad_dates.sum())} rows with invalid dates")
        df = df.loc[~bad_dates].copy()
    if df.empty:
        raise SkipDailyFile(f"{path.name}: all rows have invalid dates")

    # Keep the modal date found during discovery and drop anomalous ones.
    expected = pd.Timestamp(expected_data_date).normalize()
    wrong_date = df["data_date"].dt.normalize() != expected
    if wrong_date.any():
        print(
            f"Warning: {path.name}: discarding {int(wrong_date.sum())} rows "
            f"whose DataDate is not {expected.date()}"
        )
        df = df.loc[~wrong_date].copy()

    valid_put_call = df["put_call"].isin(["call", "put"])
    if not valid_put_call.all():
        print(f"Warning: {path.name}: discarding {int((~valid_put_call).sum())} rows with invalid PutCall")
        df = df.loc[valid_put_call].copy()

    positive = (df["strike_price"] > 0) & (df["underlying_price"] > 0)
    if not positive.all():
        print(f"Warning: {path.name}: discarding {int((~positive).sum())} rows with non-positive prices")
        df = df.loc[positive].copy()

    if df.empty:
        raise SkipDailyFile(f"{path.name}: all rows failed validation")

    df["quote_is_valid"] = (
        (df["ask_price"] > 0)
        & (df["bid_price"] >= 0)
        & (df["ask_price"] >= df["bid_price"])
    ).astype("boolean")

    df["quote_is_two_sided"] = (
        df["quote_is_valid"] & (df["bid_price"] > 0)
    ).astype("boolean")

    # An invalid quote gets NaN rather than a misleading midpoint.
    df["mid_price"] = (
        (df["bid_price"] + df["ask_price"]) / 2
    ).where(df["quote_is_valid"])

    # The derived settlement date is only meaningful for standard monthlies.
    # Applying it to a weekly would measure that option's life to the third
    # Friday of its month, which is not when it expires.
    df["is_standard_monthly"] = is_standard_monthly_expiration(df["expiration_date"])

    settlement = standard_spx_settlement_date(df["expiration_date"])
    df["settlement_date"] = settlement.where(df["is_standard_monthly"])

    df["dte_calendar_days"] = (
        (df["settlement_date"] - df["data_date"]).dt.days.astype("Int32")
    )

    # AM settlement: the option stops trading at the prior close.
    df["time_to_expiry_years"] = (
        df["dte_calendar_days"].astype("Float64") - AM_SETTLEMENT_DAY_ADJUSTMENT
    ) / DAY_COUNT_DAYS_PER_YEAR

    df["source_file"] = path.name

    return df.loc[:, DAILY_COLUMNS]

### Selecting the three horizons

For each observation date, the nearest distinct expiration to each target is
retained, ties going to the longer one. The `expiry_table` is built from
distinct `(date, expiration)` pairs so the search does not scan every quote.

In [ ]:
def select_target_expirations(
    df: pd.DataFrame,
    targets: tuple[int, ...] = TARGET_DTES,
    tolerance: int = TARGET_DTE_TOLERANCE_DAYS,
) -> tuple[pd.DataFrame, dict]:
    """Retain the nearest distinct expiration to each target DTE per date."""
    eligible = df["dte_calendar_days"].notna() & (
        df["dte_calendar_days"] >= MINIMUM_DTE_DAYS
    )
    if STANDARD_MONTHLY_ONLY:
        eligible &= df["is_standard_monthly"].fillna(False)

    expiry_table = df.loc[
        eligible, ["data_date", "expiration_date", "dte_calendar_days"]
    ].drop_duplicates()

    selections = []

    for data_date, available in expiry_table.groupby("data_date", sort=False):
        chosen_expirations = set()

        for target in targets:
            candidates = available.loc[
                ~available["expiration_date"].isin(chosen_expirations)
            ].copy()
            if candidates.empty:
                continue

            candidates["distance"] = (
                candidates["dte_calendar_days"] - target
            ).abs()
            # Prefer the longer expiration when two are equally close.
            candidates["is_shorter"] = candidates["dte_calendar_days"] < target

            choice = candidates.sort_values(
                ["distance", "is_shorter", "expiration_date"], kind="stable"
            ).iloc[0]

            if int(choice["distance"]) > tolerance:
                continue

            chosen_expirations.add(choice["expiration_date"])
            selections.append({
                "data_date": data_date,
                "expiration_date": choice["expiration_date"],
                "target_dte_days": target,
            })

    selection_table = pd.DataFrame(
        selections,
        columns=["data_date", "expiration_date", "target_dte_days"],
    )

    requested = df["data_date"].nunique() * len(targets)
    coverage = {
        "requested_chains": int(requested),
        "selected_chains": int(len(selection_table)),
    }

    if selection_table.empty:
        empty = df.iloc[0:0].copy()
        empty["target_dte_days"] = pd.Series(dtype="Int16")
        return empty.loc[:, OUTPUT_COLUMNS], coverage

    selected = df.merge(
        selection_table,
        on=["data_date", "expiration_date"],
        how="inner",
        validate="many_to_one",
    )
    selected["target_dte_days"] = selected["target_dte_days"].astype("Int16")

    return selected.loc[:, OUTPUT_COLUMNS], coverage

### Assembling one month

Two things to note in the function below, both of which were wrong in an
earlier version:

* Rows rejected for any reason -- duplicate keys, expiration before the data
  date, or an invalid quote -- are collected first and concatenated **once, at
  the end**. Building the audit frame before the last category was appended
  meant the invalid quotes never reached the `_invalid.parquet` file.
* The quality counts are taken **before** the invalid quotes are removed.
  Counting them afterwards makes "chains with 100% valid quotes" true by
  construction, which is a statistic that cannot fail and therefore says
  nothing.

In [ ]:
def prepare_month(files: list) -> tuple[pd.DataFrame, pd.DataFrame, dict]:
    """Combine, validate and reduce one month to the selected chains."""
    frames = []
    skipped_files = 0

    for path, data_date in sorted(files, key=lambda item: (item[1], item[0].name)):
        try:
            frames.append(read_and_prepare_daily_file(path, data_date))
        except SkipDailyFile as exc:
            print(f"Warning: {exc}; skipping daily file")
            skipped_files += 1

    if not frames:
        raise ValueError("No usable daily files remain for this month")

    monthly = pd.concat(frames, ignore_index=True)
    rows_read = len(monthly)

    rejected = []

    # OptionKey embeds DataDate, so a duplicate group cannot cross months.
    duplicate_keys = monthly["option_key"].duplicated(keep=False)
    if duplicate_keys.any():
        rows = monthly.loc[duplicate_keys].copy()
        rows["invalid_reason"] = "duplicate_option_key"
        rejected.append(rows)
        print(
            f"Warning: {int(duplicate_keys.sum()):,} rows "
            f"({duplicate_keys.mean() * 100:.2f}% of the month) share an "
            "option_key with a differing quote; discarding all of them"
        )
        monthly = monthly.loc[~duplicate_keys].copy()

    expired = monthly["expiration_date"] < monthly["data_date"]
    if expired.any():
        rows = monthly.loc[expired].copy()
        rows["invalid_reason"] = "expiration_before_data_date"
        rejected.append(rows)
        print(f"Warning: discarding {int(expired.sum()):,} rows expiring before their data date")
        monthly = monthly.loc[~expired].copy()

    monthly, coverage = select_target_expirations(monthly)

    # Quality counts BEFORE the invalid quotes are removed, otherwise every
    # chain is trivially 100% valid.
    valid = monthly["quote_is_valid"].fillna(False)
    two_sided = monthly["quote_is_two_sided"].fillna(False)

    chain_columns = ["data_date", "expiration_date", "target_dte_days"]
    chain_quality = (
        monthly.assign(_valid=valid, _two_sided=two_sided)
        .groupby(chain_columns, as_index=False, observed=True)
        .agg(
            quotes=("_valid", "size"),
            valid_quotes=("_valid", "sum"),
            two_sided_quotes=("_two_sided", "sum"),
        )
    )

    invalid_quotes = ~valid
    if invalid_quotes.any():
        rows = monthly.loc[invalid_quotes].copy()
        rows["invalid_reason"] = "invalid_quote"
        rejected.append(rows)
        print(f"Discarding {int(invalid_quotes.sum()):,} invalid quotes")
        monthly = monthly.loc[~invalid_quotes].copy()

    if rejected:
        invalid_data = pd.concat(rejected, ignore_index=True)
    else:
        invalid_data = monthly.iloc[0:0].copy()
        invalid_data["invalid_reason"] = pd.Series(dtype="string")

    monthly = monthly.sort_values(
        ["data_date", "target_dte_days", "expiration_date", "put_call", "strike_price"],
        kind="stable",
        ignore_index=True,
    )

    summary = {
        "files": len(files),
        "skipped_files": skipped_files,
        "rows_read": rows_read,
        "rows_kept": len(monthly),
        "rows_rejected": len(invalid_data),
        "duplicate_key_share": float(duplicate_keys.mean()) if rows_read else 0.0,
        "chains": len(chain_quality),
        "chains_fully_valid": int(
            (chain_quality["valid_quotes"] == chain_quality["quotes"]).sum()
        ),
        "valid_quote_share": float(valid.mean()) if len(valid) else float("nan"),
        "two_sided_share": float(two_sided.mean()) if len(two_sided) else float("nan"),
        **coverage,
    }

    return monthly, invalid_data, summary

In [ ]:
def write_parquet(df: pd.DataFrame, path: Path, overwrite: bool = OVERWRITE) -> None:
    """Write one compressed monthly Parquet file."""
    if path.exists() and not overwrite:
        print(f"Warning: {path} already exists; skipping write")
        return

    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_parquet(path, engine="pyarrow", compression="zstd", index=False)


def process_all_months(
    raw_dir: Path = SPX_RAW_DIR,
    processed_dir: Path = SPX_PROCESSED_DIR,
    overwrite: bool = OVERWRITE,
) -> pd.DataFrame:
    """Process every discovered month and return the per-month summary."""
    files_by_month = discover_files_by_month(raw_dir)
    summaries = []

    for (year, month), files in sorted(files_by_month.items()):
        try:
            monthly, invalid_data, summary = prepare_month(files)
        except ValueError as exc:
            print(f"{year}-{month:02d}: skipped ({exc})")
            continue

        year_dir = processed_dir / str(year)
        stem = f"spx_options_{year}_{month:02d}"

        write_parquet(monthly, year_dir / f"{stem}.parquet", overwrite)
        write_parquet(invalid_data, year_dir / f"{stem}_invalid.parquet", overwrite)

        summary |= {"year": year, "month": month}
        summaries.append(summary)

        print(
            f"{year}-{month:02d}: {summary['files']} files, "
            f"{summary['rows_kept']:,} rows kept, "
            f"{summary['rows_rejected']:,} rejected "
            f"({summary['duplicate_key_share'] * 100:.2f}% duplicate keys), "
            f"{summary['chains']} chains "
            f"({summary['selected_chains']}/{summary['requested_chains']} "
            f"date-horizon slots filled), "
            f"{summary['chains_fully_valid']} with no invalid quote"
        )

        del monthly, invalid_data

    return pd.DataFrame(summaries)


monthly_summary = process_all_months()
monthly_summary.to_parquet(PROCESSED_DIR / "monthly_processing_summary.parquet", index=False)
monthly_summary.tail()

## Coverage by year

This is the table that justifies `ANALYSIS_YEARS`. The duplicate-key share is
the reason a year gets excluded: once a large fraction of a year's rows cannot
be attributed to a single settlement convention, the chains built from what
remains are not trustworthy.

In [ ]:
yearly_coverage = (
    monthly_summary
    .groupby("year")
    .agg(
        months=("month", "nunique"),
        rows_kept=("rows_kept", "sum"),
        rows_rejected=("rows_rejected", "sum"),
        chains=("chains", "sum"),
        slots_requested=("requested_chains", "sum"),
        slots_filled=("selected_chains", "sum"),
        duplicate_key_share=("duplicate_key_share", "mean"),
        two_sided_share=("two_sided_share", "mean"),
    )
    .assign(
        slot_fill_rate=lambda t: t["slots_filled"] / t["slots_requested"],
        in_analysis=lambda t: t.index.isin(ANALYSIS_YEARS),
    )
    .round(4)
)

print("Excluded years:", EXCLUDED_YEARS)
yearly_coverage

In [ ]:
import matplotlib.pyplot as plt

from implied_stock_distributions.plotting import PALETTE, use_project_style

use_project_style()

fig, axes = plt.subplots(2, 1, figsize=(9, 5.5), sharex=True)

colors = [
    PALETTE["constrained"] if year in ANALYSIS_YEARS else PALETTE["violation"]
    for year in yearly_coverage.index
]

axes[0].bar(yearly_coverage.index, yearly_coverage["chains"], color=colors)
axes[0].set_ylabel("Chains retained")
axes[0].set_title("Coverage by year  ·  red years are excluded from the analysis")

axes[1].bar(
    yearly_coverage.index, 100 * yearly_coverage["duplicate_key_share"], color=colors
)
axes[1].set_ylabel("Duplicate option keys (%)")
axes[1].set_xlabel("Year")

plt.show()

In [ ]:
from implied_stock_distributions.config import MAX_DUPLICATE_KEY_SHARE

# The exclusion table, derived from the coverage numbers rather than typed.
exclusion_table = (
    yearly_coverage
    .assign(
        duplicate_key_pct=lambda t: (100 * t["duplicate_key_share"]).round(2),
        rule=lambda t: np.where(
            t["duplicate_key_share"] > MAX_DUPLICATE_KEY_SHARE,
            "excluded: duplicate-key share above "
            f"{100 * MAX_DUPLICATE_KEY_SHARE:.0f}%",
            "retained",
        ),
    )
    .loc[:, ["chains", "duplicate_key_pct", "slot_fill_rate", "rule", "in_analysis"]]
)

# The rule and the configured year list must agree, or one of them is stale.
derived = tuple(
    int(year) for year in exclusion_table.index
    if exclusion_table.loc[year, "rule"] == "retained"
)
if derived != tuple(y for y in ANALYSIS_YEARS if y in exclusion_table.index):
    print(
        "WARNING: config.ANALYSIS_YEARS does not match the exclusion rule.\n"
        f"  rule keeps:   {derived}\n"
        f"  config keeps: {ANALYSIS_YEARS}\n"
        "  Update config.ANALYSIS_YEARS / EXCLUDED_YEARS, or the threshold."
    )
else:
    print("config.ANALYSIS_YEARS agrees with the exclusion rule.")

exclusion_table

### What the excluded years cost

An exclusion is only defensible if its size and its direction are both
stated. The cell below puts numbers on both.

**Direction of the bias.** The dropped years are not a random sample of
market conditions. 2011 (the US downgrade and the euro crisis), 2019 and
2020 (the COVID crash and the fastest volatility spike on record) are
high-volatility years; 2012, 2013 and 2017 are unusually quiet ones, with
2017 the lowest-volatility year in modern SPX history. Losing both tails of
the volatility distribution is better than losing one, but the losses do not
cancel where it matters: the project's weakest claims are about the *left
tail* of the risk-neutral density, and the observations that discipline a
left tail are crash months. Dropping 2020 removes the single most
informative month in two decades for that question.

So the honest statement is that every tail conclusion here is estimated on a
sample from which the most extreme realisations have been removed, and the
recovered left tails are therefore **less likely to be tested against a
genuinely extreme outcome** than the full history would allow. That biases
the calibration diagnostics toward looking better than they should, not
worse.

In [ ]:
# What the dropped years would have contributed, from the same summary.
excluded_summary = (
    monthly_summary.loc[monthly_summary["year"].isin(EXCLUDED_YEARS)]
    .groupby("year")
    .agg(
        months=("month", "nunique"),
        chains=("chains", "sum"),
        rows_kept=("rows_kept", "sum"),
        duplicate_key_share=("duplicate_key_share", "mean"),
    )
)

total_chains = monthly_summary["chains"].sum()
lost = excluded_summary["chains"].sum()

print(f"Excluded years:            {EXCLUDED_YEARS}")
print(f"Chains in excluded years:  {lost:,} of {total_chains:,} "
      f"({100 * lost / total_chains:.1f}% of everything processed)")
print(f"Years retained:            {len(ANALYSIS_YEARS)} of "
      f"{len(ANALYSIS_YEARS) + len(EXCLUDED_YEARS)}")
print()

display(excluded_summary.round(4))

# 2020 month by month -- the exclusion the conclusion calls the most damaging.
month_by_month = (
    monthly_summary.loc[monthly_summary["year"].eq(2020)]
    .set_index("month")[["chains", "rows_kept", "rows_rejected",
                         "duplicate_key_share"]]
    .round(4)
)

if len(month_by_month):
    print("2020, month by month:")
    display(month_by_month)
    print(
        f"Duplicate-key share in 2020 ranges from "
        f"{month_by_month['duplicate_key_share'].min():.1%} to "
        f"{month_by_month['duplicate_key_share'].max():.1%}; the threshold is "
        f"{MAX_DUPLICATE_KEY_SHARE:.1%}."
    )
    clean = month_by_month.loc[
        month_by_month["duplicate_key_share"] <= MAX_DUPLICATE_KEY_SHARE
    ]
    print(
        f"{len(clean)} of {len(month_by_month)} months of 2020 are individually "
        "below the threshold."
    )
    if len(clean):
        print(
            "Those months could be recovered without weakening the rule -- "
            "the exclusion is applied by year only for simplicity, and "
            "nothing forces that."
        )
else:
    print("No 2020 rows in the processing summary.")

## Settlement values

Standard monthly SPX options settle against SET, the special opening
quotation published by Cboe on the third Friday. Cboe serves structured JSON
from 2020 and HTML tables before that, so both are handled. The result is
cached: re-running this notebook should not hit Cboe again unless the file is
deleted.

In [ ]:
CBOE_SET_URL = "https://www.cboe.com/index_settlement_values/get_sv_data/S/{year}/"

MONTH_PATTERN = re.compile(
    r"(January|February|March|April|May|June|July|August|September|"
    r"October|November|December)\s+(\d{4})\s+Settlement Values",
    flags=re.IGNORECASE,
)


def download_spx_set_values(years, pause: float = 0.5) -> pd.DataFrame:
    """Download standard monthly SPX (SET) settlement values from Cboe."""
    records = []

    for year in years:
        with urlopen(CBOE_SET_URL.format(year=year), timeout=30) as response:
            data = json.load(response)["data"]

        if isinstance(data, list):
            # Structured JSON, 2020 onwards.
            frame = pd.DataFrame(data)
            frame = frame.loc[
                frame["trading_symbol"].eq("SPX")
                & frame["settlement_symbol"].eq("SET")
            ]

            for row in frame.itertuples():
                date = pd.Timestamp(row.expiration_date)
                records.append({
                    "expiration_year": date.year,
                    "expiration_month": date.month,
                    "settlement_date": date,
                    "set_value": float(row.settlement_value),
                })
        else:
            # HTML tables, before 2020.
            months = MONTH_PATTERN.findall(data)
            tables = pd.read_html(StringIO(data))

            if len(months) != len(tables):
                raise ValueError(
                    f"{year}: found {len(months)} month headings but "
                    f"{len(tables)} tables; the page layout has changed."
                )

            for (month_name, year_text), table in zip(months, tables):
                labels = table.iloc[:, 0].astype(str)
                match = labels.str.contains("S&P 500 (SET)", regex=False)

                if match.sum() != 1:
                    raise ValueError(
                        f"Could not identify the SET value for {month_name} {year_text}"
                    )

                raw_value = str(table.loc[match].iloc[0, 1])
                records.append({
                    "expiration_year": int(year_text),
                    "expiration_month": pd.to_datetime(month_name, format="%B").month,
                    "settlement_date": pd.NaT,
                    "set_value": float(re.sub(r"[^0-9.-]", "", raw_value)),
                })

        time.sleep(pause)

    values = (
        pd.DataFrame(records)
        .sort_values(["expiration_year", "expiration_month"])
        .reset_index(drop=True)
    )

    duplicated = values.duplicated(["expiration_year", "expiration_month"])
    if duplicated.any():
        raise ValueError("Cboe returned more than one SET value for a month.")

    return values


if SET_VALUES_PATH.exists():
    set_values = pd.read_parquet(SET_VALUES_PATH)
    print(f"Loaded {len(set_values)} cached settlement values from {SET_VALUES_PATH}")
else:
    set_values = download_spx_set_values(SET_DOWNLOAD_YEARS)
    SET_VALUES_PATH.parent.mkdir(parents=True, exist_ok=True)
    set_values.to_parquet(SET_VALUES_PATH, index=False)
    print(f"Downloaded {len(set_values)} settlement values to {SET_VALUES_PATH}")

set_values.tail()

## Chain catalogue

The catalogue records which chains live in which monthly file, so the
analysis can iterate over chains without holding the whole sample in memory.
Paths are stored **relative to the project root**, so the catalogue works in
any clone rather than only on the machine that built it.

In [ ]:
from implied_stock_distributions.data_access import build_chain_catalog

chain_catalog = build_chain_catalog()
CHAIN_CATALOG_PATH.parent.mkdir(parents=True, exist_ok=True)
chain_catalog.to_parquet(CHAIN_CATALOG_PATH, index=False)

print(f"{len(chain_catalog):,} chains catalogued to {CHAIN_CATALOG_PATH}")
print(chain_catalog["parquet_path"].iloc[0])

chain_catalog["target_dte_days"].value_counts().sort_index()

## Realised days to expiry

The horizons are targets, not exact. This is what the buckets actually
contain, and it is why the analysis notebook reports scores per horizon
rather than pooling them.

In [ ]:
actual_dte = (
    chain_catalog.assign(
        actual_dte=(
            chain_catalog["expiration_date"] - chain_catalog["data_date"]
        ).dt.days
    )
    .groupby("target_dte_days")["actual_dte"]
    .describe(percentiles=[0.05, 0.5, 0.95])
    .round(1)
)

actual_dte

## Day count: one clock or two?

Time to expiry is used for two different things, and they do not obviously
want the same convention. **Discounting** accrues on calendar days, so
365-day years are right there. **Variance** accumulates on trading days: a
market that is shut does not move, and the run-up to Thanksgiving and
Christmas packs fewer trading days into the same calendar span than the
average.

The project uses the calendar clock for both, which is the common default.
The cell below says what that costs, by re-expressing each chain's maturity
on a trading-day clock and asking how much thirty-day implied volatility
would move. This is a robustness line, not a change: nothing downstream uses
the trading-day figure.

In [ ]:
from implied_stock_distributions.config import TRADING_DAYS_PER_YEAR

# Trading days to expiry, against the calendar-day convention in use.
sample = chain_catalog.copy()
sample["calendar_days"] = (
    sample["expiration_date"] - sample["data_date"]
).dt.days

trading_days = np.busday_count(
    sample["data_date"].to_numpy("datetime64[D]"),
    sample["expiration_date"].to_numpy("datetime64[D]"),
)
sample["trading_days"] = trading_days

calendar_tau = (
    sample["calendar_days"] - AM_SETTLEMENT_DAY_ADJUSTMENT
) / DAY_COUNT_DAYS_PER_YEAR
trading_tau = sample["trading_days"] / TRADING_DAYS_PER_YEAR

# Implied volatility scales as 1/sqrt(tau) for a fixed total variance, so
# the ratio of clocks maps directly to a ratio of volatilities.
sample["iv_ratio"] = np.sqrt(calendar_tau / trading_tau)

summary = (
    sample.groupby("target_dte_days")["iv_ratio"]
    .describe(percentiles=[0.01, 0.5, 0.99])
    .round(4)
)

print("Ratio of trading-clock to calendar-clock implied volatility")
print("(1.000 would mean the two conventions agree)")
display(summary)

worst = sample.loc[sample["iv_ratio"].sub(1).abs().idxmax()]
print(
    f"Largest single discrepancy: {abs(worst['iv_ratio'] - 1):.2%} on "
    f"{worst['data_date'].date()} into {worst['expiration_date'].date()} "
    f"({int(worst['calendar_days'])} calendar days, "
    f"{int(worst['trading_days'])} trading days)."
)
print(
    "At a thirty-day implied volatility near 16%, a 3% relative shift is "
    "about half a volatility point -- concentrated, as expected, in the "
    "holiday periods."
)